# ED Agent Mesh — ECG + Triage Standing Orders (Patched)
Run top‑to‑bottom. Logs show actions, blocks, and ECG traffic‑light colors.

In [ ]:

# --- Stage 1 Safety Guardrails (compact) ---
import pandas as pd
pd.options.mode.chained_assignment = 'raise'  # loud on chained indexing

def load_ed_data(path_or_buf):
    df = pd.read_csv(path_or_buf).convert_dtypes()
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], utc=True, errors='raise')
    if 'encounter_id' in df.columns:
        assert df['encounter_id'].is_unique, "Duplicate encounter_id detected!"
    return df

def validate_df(df, name="DataFrame"):
    if 'encounter_id' in df.columns:
        assert df['encounter_id'].is_unique, f"{name}: duplicate encounter_id"
    if 'ts' in df.columns:
        assert df['ts'].is_monotonic_increasing, f"{name}: timestamps not monotonic"
    print(f"[OK] {name} passed Stage 1 checks.")

# LLM proposal minimal validator (allowed actions + required fields)
import json, hashlib
from typing import Dict, Any, List

ALLOWED_ACTIONS = {
    "page_team","order_ct","request_labs","hold_bed","open_case","order_ecg","ed_hold",
    "icu_downgrade","bed.request",
    # new ECG/triage actions
    "request_vbga","request_abga","ecg_interpret","ecg_alert","ecg_repeat","ecg_context_fetch"
}

PARAM_SCHEMAS: Dict[str, Dict[str, str]] = {
    "page_team":    {"team": "str", "priority": "enum:STAT|URGENT", "reason": "str"},
    "order_ct":     {"protocol": "str", "priority": "enum:STAT|ROUTINE"},
    "request_labs": {"panel_id": "str", "priority": "enum:STAT|ROUTINE"},
    "hold_bed":     {"service": "enum:ICU|CARDS|NEPHRO|OBGYN|MED", "level": "enum:WARD|STEPDOWN|ICU"},
    "open_case":    {"patient_ref": "str"},
    "order_ecg":    {"priority": "enum:STAT|ROUTINE"},
    "ed_hold":      {"reason": "str"},
    "icu_downgrade":{"to_level": "enum:WARD|STEPDOWN"},
    # new
    "request_vbga": {"site": "enum:venous"},
    "request_abga": {"site": "enum:arterial"},
    "ecg_interpret":{"ecg_id": "str", "source": "enum:triage|ed|ems"},
    "ecg_alert":    {"severity": "enum:CRITICAL|URGENT", "phenotype": "str", "confidence": "float"},
    "ecg_repeat":   {"minutes": "int"},
    "ecg_context_fetch": {"window": "enum:recent|mid|long"}
}

MUST_REQUIRE_PERMIT = {"order_ct", "request_labs", "hold_bed", "icu_downgrade"}  # gases & ECG alerts not in this list

def _ensure_type(name: str, val: Any, want: str):
    if want == "str" and not isinstance(val, str):
        raise ValueError(f"param.{name} must be str")
    if want == "int" and not isinstance(val, int):
        raise ValueError(f"param.{name} must be int")
    if want == "float" and not isinstance(val, (int, float)):
        raise ValueError(f"param.{name} must be float")

def _ensure_enum(name: str, val: Any, options: List[str]):
    if not isinstance(val, str) or val not in options:
        raise ValueError(f"param.{name} must be one of {options}")

def _validate_params(action: str, params: Dict[str, Any]) -> Dict[str, Any]:
    if action not in PARAM_SCHEMAS:
        if params:
            raise ValueError(f"no schema for action '{action}', params must be empty for now")
        return {}
    schema = PARAM_SCHEMAS[action]
    extra = set(params.keys()) - set(schema.keys())
    if extra:
        raise ValueError(f"unexpected params for {action}: {sorted(extra)}")
    missing = [k for k in schema.keys() if k not in params]
    if missing:
        raise ValueError(f"missing params for {action}: {missing}")
    for k, rule in schema.items():
        if rule.startswith("enum:"):
            options = rule.split(":",1)[1].split("|")
            _ensure_enum(k, params[k], options)
        else:
            _ensure_type(k, params[k], rule)
    return params

def _hash_params(encounter_id: str, action: str, params: Dict[str, Any]) -> str:
    blob = json.dumps({"encounter_id": encounter_id, "action": action, "params": params}, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def parse_and_validate_proposal(raw: str, encounter_id: str) -> Dict[str, Any]:
    d = json.loads(raw)
    for key in ("action", "params", "requires_permit", "justification"):
        if key not in d:
            raise ValueError(f"missing required field: {key}")
    if d["action"] not in ALLOWED_ACTIONS:
        raise ValueError(f"action '{d['action']}' not allowed")
    d["params"] = _validate_params(d["action"], d["params"])
    if d["action"] in MUST_REQUIRE_PERMIT and d["requires_permit"] is not True:
        raise ValueError(f"action '{d['action']}' must require permit")
    return d

print("[Stage 1 loaded: pandas guardrails + expanded LLM validator]")


In [ ]:

# ==== Policy Layer Patch (cooldowns, idempotency, edge-trigger, repeats) ====
from typing import Callable, Dict, Any, Tuple
from collections import defaultdict
import time, json, hashlib

def log_event(kind: str, action: str = "", note: str = "") -> None:
    print(f"{kind:<6} | {action} | {note}")

CANON = {
    "proposal.order.ecg":       "order_ecg",
    "proposal.order.labs":      "request_labs",
    "proposal.order.ct":        "order_ct",
    "proposal.bed.request":     "bed.request",
    "proposal.ed_hold":         "ed_hold",
    "proposal.icu_downgrade":   "icu_downgrade",
    # new names
    "proposal.request.vbga":    "request_vbga",
    "proposal.request.abga":    "request_abga",
    "proposal.ecg.interpret":   "ecg_interpret",
    "proposal.ecg.alert":       "ecg_alert",
    "proposal.ecg.repeat":      "ecg_repeat",
    "proposal.ecg.context":     "ecg_context_fetch"
}

COOLDOWN_SEC = {
    "bed.request":   30 * 60,
    "ed_hold":       10 * 60,
    "order_ct":      365 * 24 * 3600,
    "order_ecg":     2 * 3600,
    "request_labs":  2 * 3600,
    "icu_downgrade": 60 * 60,
    # new
    "request_vbga":  60 * 60,
    "request_abga":  60 * 60,
    "ecg_alert":     30 * 60,
    "ecg_repeat":    15 * 60
}

REPEAT_POLICY = {
    "order_ct":     {"allow": False},
    "order_ecg":    {"allow": True, "interval_sec": 2 * 3600},
    "request_labs": {"allow": False},
    "request_vbga": {"allow": True, "interval_sec": 60 * 60},
    "request_abga": {"allow": True, "interval_sec": 60 * 60},
    "ecg_repeat":   {"allow": True, "interval_sec": 15 * 60}
}

_last_fired     = defaultdict(float)
_completed      = set()
_last_done      = {}
awaiting_capacity = defaultdict(bool)

def _idem_key(enc: str, canon: str, params: Dict[str, Any]) -> str:
    blob = json.dumps({"e": enc, "a": canon, "p": params}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()

def _params_sig(params: Dict[str, Any]) -> str:
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()

def on_capacity_change(encounter_id: str, icu_free_beds: int) -> None:
    awaiting_capacity[encounter_id] = (icu_free_beds == 0)

def _can_emit(enc: str, canon: str, params: Dict[str, Any], now=None) -> Tuple[bool, str, str]:
    now = now or time.time()
    key = _idem_key(enc, canon, params)
    if key in _completed:
        return False, "already completed", key
    cd = COOLDOWN_SEC.get(canon, 0)
    if now - _last_fired[key] < cd:
        return False, f"cooldown {int(cd - (now - _last_fired[key]))}s", key
    if canon == "bed.request" and awaiting_capacity.get(enc, False):
        return False, "awaiting capacity change", key
    return True, "ok", key

def _can_repeat(enc: str, canon: str, params: Dict[str, Any], ctx: Dict[str, Any], now=None) -> Tuple[bool, str]:
    now = now or time.time()
    rule = REPEAT_POLICY.get(canon, {"allow": False})
    sig  = _params_sig(params)
    rk   = (enc, canon, sig)
    if canon == "order_ct" and ctx.get("new_indication", False):
        return True, "new indication"
    if not rule.get("allow", False):
        return (rk not in _last_done), "repeat not allowed"
    interval = rule.get("interval_sec")
    if "by_panel" in rule:
        interval = rule["by_panel"].get(params.get("panel_id"), interval)
    last = _last_done.get(rk, 0)
    if (now - last) < (interval or 0):
        return False, f"repeat cooldown {int((interval or 0) - (now - last))}s"
    return True, "ok"

def policy_emit(encounter_id: str, action: str, params: Dict[str, Any],
                ctx: Dict[str, Any], do_emit: Callable[[str, Dict[str, Any]], None]) -> bool:
    canon = CANON.get(action, action)
    if canon == "bed.request" and ctx.get("icu_free_beds", 0) == 0:
        awaiting_capacity[encounter_id] = True
    ok1, reason = _can_repeat(encounter_id, canon, params, ctx)
    ok2, why2, key = _can_emit(encounter_id, canon, params)
    if not ok1:
        log_event("block", action, reason); return False
    if not ok2:
        log_event("block", action, why2);   return False
    do_emit(action, params)
    _last_fired[key] = time.time()
    log_event("action", action, "emitted")
    return True

def policy_complete(encounter_id: str, action: str, params: Dict[str, Any]) -> None:
    canon = CANON.get(action, action)
    key   = _idem_key(encounter_id, canon, params)
    _completed.add(key)
    _last_done[(encounter_id, canon, _params_sig(params))] = time.time()
    log_event("audit", action, "completed")

print("[Policy Layer ready: cooldowns, idempotency, edge-trigger, repeats]")


In [ ]:

# --- Event Bus + raw_emit + Capacity store ---
from collections import deque

class EventBus:
    def __init__(self):
        self.queue = deque()
        self.log = []
    def propose(self, action, params):
        self.queue.append((action, params))
        self.log.append(("propose", action, params))
        log_event("emit", action, str(params))

eventbus = EventBus()

def raw_emit(action, params):
    eventbus.propose(action, params)

class Capacity:
    def __init__(self, icu_total=2, icu_occupied=2):
        self.icu_total = icu_total
        self.icu_occupied = icu_occupied
    @property
    def icu_free(self):
        return max(0, self.icu_total - self.icu_occupied)

capacity = Capacity(icu_total=2, icu_occupied=2)


In [ ]:

# --- ECG Context, Dynamics, and Traffic-Light helper ---
def get_prior_ecg_summary(encounter_id):
    # synthetic prior ECG summary for demo
    return {"when": "2025-07-29T14:10Z", "metrics": {"QTc": 430}, "st_mm_by_lead": {"V2": 0.0, "V3": 0.0}, "phenotype":"NORMAL"}

def compute_ecg_deltas(current, prior):
    if not prior:
        return {"delta_st_mm_max": 0.0, "reciprocal_st_mm_max": 0.0, "unchanged_vs_prior": None}
    d_v2 = current.get("st_mm_by_lead",{}).get("V2",0.0) - prior.get("st_mm_by_lead",{}).get("V2",0.0)
    d_v3 = current.get("st_mm_by_lead",{}).get("V3",0.0) - prior.get("st_mm_by_lead",{}).get("V3",0.0)
    delta_st = max(d_v2, d_v3)
    recip = current.get("reciprocal_st_mm_max", 0.0)
    unchanged = (abs(delta_st) < 0.5 and recip < 0.5)
    return {"delta_st_mm_max": float(round(delta_st,2)),
            "reciprocal_st_mm_max": float(round(recip,2)),
            "unchanged_vs_prior": unchanged}

def is_abnormal_vitals(vitals):
    return (vitals.get("MAP", 80) < 65) or (vitals.get("SpO2", 97) < 92) or (vitals.get("HR", 80) > 130) or (vitals.get("HR",80) < 40)

def ecg_traffic_light(ai, compare, ctx, troponin, cfg):
    # Absolute reds
    if ai["metrics"].get("vt_vf"):
        return "RED", ["VT/VF"], "Page attending"
    if ai["metrics"].get("chb") and ai["metrics"].get("brady_hypotension"):
        return "RED", ["Complete heart block + hypotension/LOC"], "Page attending"
    if ctx.get("shock"):
        if ai["phenotype"] == "HyperK" or (ctx.get("k_value") is not None and ctx["k_value"] >= 6.5):
            return "RED", ["HyperK pattern + shock/↑K"], "Page attending"

    meets_static = (ai["severity"] == "CRITICAL")
    dynamic = (compare.get("delta_st_mm_max", 0.0) >= cfg["delta_st_mm"] or
               compare.get("reciprocal_st_mm_max", 0.0) >= cfg["recip_mm"])
    context_ok = (ctx.get("acs_symptoms") or ctx.get("abnormal_vitals"))
    conf_ok = (ai.get("confidence", 0.0) >= cfg["min_conf"])
    confounder = ai.get("confounder", "none")

    if meets_static and dynamic and context_ok and conf_ok:
        if confounder in {"lbbb", "paced", "lvh_strain", "early_repol", "pericarditis"}:
            if troponin.get("delta_positive") is True:
                return "RED", ["Occlusion + dynamics + context + Δtroponin"], "Page attending"
        else:
            return "RED", ["Occlusion + dynamics + context"], "Page attending"

    if ai["metrics"].get("HR", 0) > 150 and ai["phenotype"] in {"AF_RVR","Tachy"}:
        return "YELLOW", ["AF with RVR >150"], "Show resident"
    if ai["metrics"].get("QTc", 0) >= 500:
        return "YELLOW", ["QTc ≥ 500 ms"], "Show resident"
    if ai["phenotype"] in {"Wellens","Brugada","PacerFailure","NewLBBB"}:
        return "YELLOW", [ai["phenotype"]], "Show resident"
    if meets_static or ai["severity"] == "URGENT":
        return "YELLOW", ["Abnormal ECG—needs review"], "Show resident"
    if compare.get("unchanged_vs_prior") is False:
        return "YELLOW", ["New changes vs prior but not RED"], "Show resident"
    if troponin.get("delta_positive") is None:
        return "YELLOW", ["Troponin Δ unknown—review"], "Show resident"

    return "GREEN", ["Normal/benign; unchanged; low risk"], "Log only"


In [ ]:

# --- Triage standing orders & ECG flow ---
def triage_standing_orders(encounter_id, vitals, context_flags):
    # Auto actions at triage: ECG, big labs, vBGA always; aBGA if A/B problem or critical C
    ctx = {
        "identity_bound": True,
        "two_identifiers_checked": True,
        "barcode_label_printed": True,
        "airway_compromise": context_flags.get("A", False),
        "severe_resp_distress": context_flags.get("B", False),
        "spo2_room_air": vitals.get("SpO2", 97),
        "shock_or_map_lt_65": (vitals.get("MAP", 80) < 65) or context_flags.get("C_critical", False),
        "icu_free_beds": capacity.icu_free
    }
    # Sync edge-trigger for capacity at start
    on_capacity_change(encounter_id, capacity.icu_free)

    policy_emit(encounter_id, "proposal.order.ecg", {"priority":"STAT"}, ctx, raw_emit)
    policy_emit(encounter_id, "proposal.order.labs", {"panel_id":"ed_big_panel","priority":"STAT"}, ctx, raw_emit)
    policy_emit(encounter_id, "proposal.request.vbga", {"site":"venous"}, ctx, raw_emit)

    if (ctx["airway_compromise"] or ctx["severe_resp_distress"] or ctx["spo2_room_air"] <= 92 or ctx["shock_or_map_lt_65"]):
        policy_emit(encounter_id, "proposal.request.abga", {"site":"arterial"}, ctx, raw_emit)

def on_ecg_result(encounter_id, ecg_ai, vitals, troponin_info):
    prior = get_prior_ecg_summary(encounter_id)
    compare = compute_ecg_deltas(ecg_ai, prior)
    ctx = {
        "acs_symptoms": True if ecg_ai.get("symptoms","") == "chest_pain" else False,
        "abnormal_vitals": is_abnormal_vitals(vitals),
        "k_value": troponin_info.get("k_value"),
        "shock": vitals.get("MAP",80) < 65
    }

    policy_emit(encounter_id, "proposal.ecg.interpret",
                {"ecg_id": ecg_ai.get("ecg_id","ecg0"), "source": ecg_ai.get("source","triage")},
                {"icu_free_beds": capacity.icu_free}, raw_emit)

    cfg = {"min_conf": 0.85, "delta_st_mm": 1.0, "recip_mm": 0.5}
    color, why, action = ecg_traffic_light(ecg_ai, compare, ctx, troponin_info, cfg)
    log_event("ECG  ", color, "; ".join(why))

    if color == "RED":
        policy_emit(encounter_id, "proposal.ecg.alert",
                    {"severity":"CRITICAL","phenotype":ecg_ai["phenotype"],"confidence":ecg_ai["confidence"]},
                    {"icu_free_beds": capacity.icu_free}, raw_emit)
    elif color == "YELLOW":
        policy_emit(encounter_id, "proposal.ecg.alert",
                    {"severity":"URGENT","phenotype":ecg_ai["phenotype"],"confidence":ecg_ai["confidence"]},
                    {"icu_free_beds": capacity.icu_free}, raw_emit)


In [ ]:

# --- Synthetic scenarios ---
def run_scenario_green():
    print("\n=== Scenario GREEN: benign ECG, unchanged vs prior ===")
    enc = "ED-ECG-001"
    vitals = {"HR":84, "MAP":78, "SpO2":97}
    triage_standing_orders(enc, vitals, context_flags={"A":False,"B":False,"C_critical":False})
    ecg_ai = {
        "ecg_id":"ecg_green", "source":"triage", "phenotype":"NORMAL", "severity":"ROUTINE",
        "confidence":0.92, "confounder":"none", "st_mm_by_lead":{"V2":0.0,"V3":0.0},
        "metrics":{"HR":84, "QTc":430}
    }
    troponin = {"assay":"hsTnT","delta_positive":False, "k_value":4.2}
    on_ecg_result(enc, ecg_ai, vitals, troponin)
    print("— done GREEN —")

def run_scenario_yellow_qtc():
    print("\n=== Scenario YELLOW: QTc prolongation ===")
    enc = "ED-ECG-002"
    vitals = {"HR":88, "MAP":80, "SpO2":98}
    triage_standing_orders(enc, vitals, context_flags={"A":False,"B":False,"C_critical":False})
    ecg_ai = {
        "ecg_id":"ecg_yellow", "source":"triage", "phenotype":"QTcProlonged", "severity":"URGENT",
        "confidence":0.90, "confounder":"none", "st_mm_by_lead":{"V2":0.0,"V3":0.0},
        "metrics":{"HR":88, "QTc":520}
    }
    troponin = {"assay":"hsTnT","delta_positive":False, "k_value":4.1}
    on_ecg_result(enc, ecg_ai, vitals, troponin)
    print("— done YELLOW —")

def run_scenario_red_dynamic_stemi():
    print("\n=== Scenario RED: dynamic anterior occlusion + context ===")
    enc = "ED-ECG-003"
    capacity.icu_occupied = 2  # ICU full at start
    vitals = {"HR":96, "MAP":72, "SpO2":94}
    triage_standing_orders(enc, vitals, context_flags={"A":False,"B":True,"C_critical":False})
    ecg_ai = {
        "ecg_id":"ecg_red", "source":"triage", "phenotype":"STEMI", "severity":"CRITICAL",
        "confidence":0.91, "confounder":"none",
        "st_mm_by_lead":{"V2":1.5,"V3":2.0},
        "reciprocal_st_mm_max":0.6,
        "symptoms":"chest_pain",
        "metrics":{"HR":96, "QTc":470, "vt_vf":False, "chb":False, "brady_hypotension":False}
    }
    troponin = {"assay":"hsTnT","delta_positive":True, "k_value":4.3}
    on_ecg_result(enc, ecg_ai, vitals, troponin)
    print("— done RED —")

# Run all
run_scenario_green()
run_scenario_yellow_qtc()
run_scenario_red_dynamic_stemi()
